# Chapter 7 — 지시를 따르도록 미세 조정하기(Instruction Fine-tuning) : 코드 해설 노트북 (Python 초보용)

> `Chapter_7_Exercise_Follow_Instructions.ipynb` 의 원본 코드 셀은 그대로 두고, 번호가 붙은 셀마다 바로 위에 설명을 넣은 해설판입니다. 셀 1이 아주 길어서 설명을 **함수 단위**로 나누었습니다.

## 이 챕터에서 하는 일 (한 문장)

"글을 이어 쓰기만 하던" GPT-2 를 **"시키는 일(지시)에 답하는" 모델**로 바꿉니다. ChatGPT 같은 대화형 모델로 가는 마지막 단계입니다.

## 무엇이 달라지나

```
 pretrained GPT-2 :   "Edit the following sentence for grammar."
                       -> just continues writing similar-looking text

 after this chapter : "Edit the following sentence for grammar.  He go to the park every day."
                       -> "He goes to the park every day."
```
사전 학습된 모델은 지시문을 봐도 그것을 **지시로 이해하지 못하고** 비슷한 글을 이어 쓸 뿐입니다. 그래서 **(지시, 입력, 모범 답변)** 묶음을 많이 보여 주어 "이런 형식이 나오면 이렇게 답하는 것"이라고 가르칩니다. 이것을 **지도 미세 조정(SFT, Supervised Fine-Tuning)** 이라고도 부릅니다.

## 데이터는 이렇게 생겼습니다

`instruction-data.json` 에는 아래와 같은 항목이 1,100개 들어 있습니다.

```json
{
  "instruction": "Edit the following sentence for grammar.",
  "input": "He go to the park every day.",
  "output": "He goes to the park every day."
}
```
| 키 | 뜻 |
|---|---|
| `instruction` | 시키는 일 |
| `input` | 그 일의 대상 (없을 수도 있음 → 빈 문자열 `""`) |
| `output` | 모범 답변 |

## Chapter 6 과 무엇이 다른가

| | Chapter 6 (분류) | Chapter 7 (지시 따르기) |
|---|---|---|
| 모델이 내놓는 것 | 라벨 1개 (0 또는 1) | **문장** (토큰 여러 개) |
| 출력 헤드 | 2개짜리로 교체 | **그대로** (50257개) |
| 손실 계산 | 마지막 토큰만 | **모든 토큰** (Chapter 5 방식) |
| 학습 범위 | 일부만 풀거나 LoRA | 전체 가중치 |
| 길이 맞추기 | 데이터 전체를 같은 길이로 | **배치마다** 그 배치의 최대 길이로 |

즉 학습 방식 자체는 **Chapter 5 의 "다음 토큰 맞히기"와 같고**, 데이터의 생김새만 "지시 + 답변" 형식으로 바뀐 것입니다.

## 셀 구성

| 셀 | 내용 |
|---|---|
| 1 | 클래스·함수 정의 : `InstructionDataset`, `custom_collate_fn` ⭐, `download_and_load_file`, `format_input` ⭐, `plot_losses`, `main` |
| 2 | `main(test_mode=False)` 실행 |

### ⚠️ 실행 전 확인 — 폴더 이름
이 노트북의 코드는 데이터를 `datas/instruction-data.json` 에서 읽고, 결과를 `outputs/` 폴더에 저장하도록 되어 있습니다. 그런데 현재 작업 폴더에는 **`data/`** 와 **`output/`** 폴더만 있습니다(끝에 s 가 없음). 그대로 실행하면 파일을 저장하는 줄에서 `FileNotFoundError` 가 날 수 있으니, 실행 전에 `datas`, `outputs` 폴더를 만들거나 코드의 경로를 `data/`, `output/` 으로 고쳐서 사용하세요.

---
## 셀 1 — 클래스와 함수 정의

### 🎯 이 셀이 하는 일
지시 미세 조정에 필요한 모든 것을 **정의**합니다. 실행해도 출력은 없습니다.

| 이름 | 한 줄 역할 |
|---|---|
| `format_input` | 데이터 한 항목을 **정해진 프롬프트 양식**의 글로 바꿈 ⭐ |
| `InstructionDataset` | 모든 항목을 "프롬프트 + 답변" 글로 만들어 토큰화해 보관 |
| `custom_collate_fn` | 길이가 다른 샘플들을 **한 배치로 묶는 방법** 정의 ⭐ |
| `download_and_load_file` | JSON 데이터 내려받아 읽기 |
| `plot_losses` | 손실 그래프 저장 |
| `main` | 전 과정 실행 |

> 읽는 순서 : 코드에는 `InstructionDataset` 이 먼저 나오지만, 그 안에서 `format_input` 을 쓰므로 **`format_input` 부터** 설명합니다. (파이썬은 함수를 실제로 부르는 시점에만 정의되어 있으면 되므로, 코드상의 순서는 문제가 되지 않습니다.)

### 새로 등장하는 import

| 코드 | 용도 |
|---|---|
| `from functools import partial` | 함수의 인자 일부를 **미리 채워 둔** 새 함수를 만드는 도구 |
| `from importlib.metadata import version` | 설치된 패키지의 버전 확인 |
| `import json` | JSON 파일 읽고 쓰기 |
| `import re` | 정규 표현식 (글자 패턴 찾아 바꾸기) |
| `from tqdm import tqdm` | 반복문에 **진행률 막대** 표시 |
| `from previous_chapters import (...)` | Chapter 4~5 에서 만든 `GPTModel`, `generate`, `train_model_simple`, `calc_loss_loader`, `text_to_token_ids`, `token_ids_to_text`, 그리고 `load_gpt2_model` |

---
### ① `format_input` — 프롬프트 양식 만들기 (⭐)

```python
def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )
    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""
    return instruction_text + input_text
```
- `entry` : 데이터 한 항목(딕셔너리). `entry['instruction']` 으로 지시문을 꺼냅니다.
- 괄호 안에 f-문자열 세 개를 나란히 쓰면 **하나로 이어 붙습니다.**
- `\n` 은 줄바꿈, `\n\n` 은 빈 줄 하나.
- `A if 조건 else B` : `entry["input"]` 이 **비어 있지 않으면** Input 부분을 만들고, 빈 문자열이면 `""`. (파이썬에서 빈 문자열은 거짓으로 취급됩니다)

결과물 :
```
Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Edit the following sentence for grammar.

### Input:
He go to the park every day.
```
이 양식을 **Alpaca 프롬프트 형식**이라고 합니다. `### Instruction:`, `### Input:`, `### Response:` 같은 **고정된 표지판**을 항상 같은 자리에 넣어 주면, 모델이 "여기까지가 문제, 여기부터가 답"이라는 구조를 쉽게 배웁니다. **학습할 때와 사용할 때 똑같은 양식**을 써야 한다는 점이 중요합니다.

---
### ② `InstructionDataset`

```python
def __init__(self, data, tokenizer):
    self.data = data
    self.encoded_texts = []
    for entry in data:
        instruction_plus_input = format_input(entry)
        response_text = f"\n\n### Response:\n{entry['output']}"
        full_text = instruction_plus_input + response_text
        self.encoded_texts.append(tokenizer.encode(full_text))
```
항목마다 **프롬프트 + `### Response:` + 모범 답변**을 하나의 글로 이어 붙인 뒤 토큰화해서 보관합니다.

```
 [ prompt ............ ][ ### Response: ][ answer ...... ]
 |<---------------- one training text ------------------>|
```
Chapter 5 처럼 "다음 토큰 맞히기"로 학습하면, 모델은 자연스럽게 **프롬프트 뒤에 어떤 답이 이어져야 하는지**를 배우게 됩니다.

```python
def __getitem__(self, index):
    return self.encoded_texts[index]

def __len__(self):
    return len(self.data)
```
Chapter 2 의 데이터셋과 달리 **(입력, 정답) 쌍을 만들지 않고 토큰 리스트 하나만** 돌려줍니다. 쌍으로 만드는 일과 길이 맞추는 일은 다음의 `custom_collate_fn` 이 배치를 묶을 때 합니다.

---
### ③ `custom_collate_fn` — 배치 묶는 방법 (⭐ 이 챕터에서 가장 까다로운 부분)

**collate 함수란?** DataLoader 가 샘플 여러 개를 꺼낸 뒤 **하나의 배치 텐서로 합칠 때 부르는 함수**입니다. 기본 방식은 샘플 길이가 모두 같아야 하는데, 여기서는 지시문마다 길이가 제각각이라 **직접 만든 방식**이 필요합니다.

이 함수가 하는 일 5가지 :
1. 끝에 "글 끝" 토큰(50256) 붙이기
2. 배치에서 가장 긴 것에 맞춰 패딩하기
3. 한 칸 밀어서 입력/정답 만들기
4. 정답의 **남는 패딩을 −100 으로 바꿔 채점에서 제외**하기
5. 텐서로 묶어 장치로 보내기

**인자**
```python
def custom_collate_fn(batch, pad_token_id=50256, ignore_index=-100,
                      allowed_max_length=None, device="cpu"):
```
`batch` 는 토큰 리스트들의 리스트입니다. 예) `[[1, 2, 3], [4, 5]]`

**단계별로 따라가기** — `batch = [[1, 2, 3], [4, 5]]` 로 직접 계산해 봅니다.

```python
batch_max_length = max(len(item)+1 for item in batch)
```
각 샘플 길이에 1(끝 토큰 자리)을 더한 값 중 최댓값 → `max(4, 3)` = **4**

```python
for item in batch:
    new_item = item.copy()
    new_item += [pad_token_id]
    padded = new_item + [pad_token_id] * (batch_max_length - len(new_item))
```
- `.copy()` : 원본 리스트를 건드리지 않으려고 **복사본**을 만듭니다.
- `+= [50256]` : 끝에 "글 끝" 토큰 추가
- 모자란 만큼 50256 으로 더 채움

```
 [1, 2, 3]  ->  [1, 2, 3, 50256]          (already length 4)
 [4, 5]     ->  [4, 5, 50256]  ->  [4, 5, 50256, 50256]
```

```python
    inputs = torch.tensor(padded[:-1])
    targets = torch.tensor(padded[1:])
```
- `padded[:-1]` : 마지막 하나를 뺀 전부
- `padded[1:]` : 첫 번째를 뺀 전부 → **한 칸 민 정답** (Chapter 2 와 같은 원리)

```
 sample 1   inputs  [1, 2, 3]            targets [2, 3, 50256]
 sample 2   inputs  [4, 5, 50256]        targets [5, 50256, 50256]
```

```python
    mask = targets == pad_token_id
    indices = torch.nonzero(mask).squeeze()
    if indices.numel() > 1:
        targets[indices[1:]] = ignore_index
```
- `targets == pad_token_id` : 정답에서 50256 인 자리를 True 로 표시
- `torch.nonzero(mask)` : True 인 **위치 번호**들을 찾음. `.squeeze()` 로 한 줄로 정리
- `indices.numel() > 1` : 50256 이 **2개 이상**일 때만
- `indices[1:]` : **첫 번째 50256 은 빼고** 그 뒤의 것들만 → −100 으로 바꿈

```
 sample 1   targets [2, 3, 50256]             only one 50256 -> keep
 sample 2   targets [5, 50256, 50256]
                        keep    -> -100
            targets [5, 50256, -100]
```

**⭐ 왜 이렇게 할까?**
- **−100 의 의미** : PyTorch 의 `cross_entropy` 는 정답이 **−100 인 자리는 채점하지 않고 건너뜁니다**(기본 약속값). 패딩은 길이를 맞추려고 넣은 가짜 토큰이라, "패딩을 맞히는 법"을 배우게 하면 안 됩니다.
- **첫 번째 50256 을 남기는 이유** : 그것은 패딩이 아니라 **"답변이 여기서 끝난다"는 진짜 신호**입니다. 이것을 정답으로 남겨 둬야 모델이 **답을 다 쓰면 멈추는 법**(끝 토큰 출력하기)을 배웁니다. 이것을 배우지 못하면 답을 한 뒤에도 끝없이 글을 이어 씁니다.

```python
    if allowed_max_length is not None:
        inputs = inputs[:allowed_max_length]
        targets = targets[:allowed_max_length]
    inputs_lst.append(inputs)
    targets_lst.append(targets)
```
너무 긴 샘플은 모델이 볼 수 있는 최대 길이(1024)로 자릅니다.

```python
inputs_tensor = torch.stack(inputs_lst).to(device)
targets_tensor = torch.stack(targets_lst).to(device)
return inputs_tensor, targets_tensor
```
`torch.stack` : 같은 길이의 텐서들을 쌓아 `[배치 크기, 길이]` 표로 만들고 GPU 로 보냅니다.

**최종 결과**
```
 inputs  = [[1, 2, 3    ],        targets = [[2, 3,     50256],
            [4, 5, 50256]]                   [5, 50256, -100 ]]
```

**Chapter 6 과의 차이** : Chapter 6 은 데이터 **전체**를 가장 긴 메시지 길이로 맞췄지만, 여기서는 **배치마다** 그 배치에서 가장 긴 것에만 맞춥니다. 짧은 샘플끼리 모인 배치는 짧게 끝나므로 계산 낭비가 줄어듭니다.

---
### ④ `download_and_load_file`

```python
if not os.path.exists(file_path):
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    text_data = response.text
    with open(file_path, "w", encoding="utf-8") as file:
        file.write(text_data)

with open(file_path, "r", encoding="utf-8") as file:
    data = json.load(file)
return data
```
파일이 없으면 내려받아 저장하고, 그다음 `json.load` 로 읽습니다. JSON 파일을 읽으면 파이썬의 **리스트와 딕셔너리**로 바뀝니다. 여기서 `data` 는 "딕셔너리 1,100개가 담긴 리스트"입니다.

---
### ⑤ `plot_losses`
Chapter 5 의 것과 같은 구조입니다(아래 x축 = 에포크, 위 x축 = 본 토큰 수). 그림을 `outputs/loss-plot-standalone.pdf` 로 저장합니다.

---
### ⑥ `main(test_mode=False)` — 전 과정

**0. 패키지 버전 출력**
```python
pkgs = ["matplotlib", "tiktoken", "torch", "tqdm"]
for p in pkgs:
    print(f"{p} version: {version(p)}")
print(50*"-")
```
문제가 생겼을 때 환경을 확인하기 위한 출력입니다.

**1. 데이터 준비와 나누기**
```python
data = download_and_load_file(file_path, url)
train_portion = int(len(data) * 0.85)
test_portion = int(len(data) * 0.1)

train_data = data[:train_portion]
test_data = data[train_portion:train_portion + test_portion]
val_data = data[train_portion + test_portion:]
```
1,100개를 앞에서부터 **학습 85%(935개) / 테스트 10%(110개) / 검증 나머지 5%(55개)** 로 자릅니다.

```python
if test_mode:
    train_data = train_data[:10]
    ...
```
`test_mode=True` 면 10개씩만 써서 코드가 돌아가는지만 빠르게 확인합니다.

**`partial` 로 collate 함수 맞춤 제작**
```python
customized_collate_fn = partial(custom_collate_fn, device=device, allowed_max_length=1024)
```
DataLoader 는 collate 함수를 부를 때 **`batch` 하나만** 넘겨줍니다. 그런데 우리는 `device` 와 `allowed_max_length` 도 정해 주고 싶습니다. `partial(함수, 인자=값, ...)` 은 **일부 인자를 미리 채워 넣은 새 함수**를 만들어 줍니다.

```python
# 아래 두 줄은 같은 일을 합니다
customized_collate_fn(batch)
custom_collate_fn(batch, device=device, allowed_max_length=1024)
```

**데이터 로더**
```python
train_dataset = InstructionDataset(train_data, tokenizer)
train_loader = DataLoader(
    train_dataset, batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=True, drop_last=True, num_workers=num_workers
)
```
`collate_fn=...` 으로 "배치를 묶을 때 이 함수를 써라"라고 지정합니다. 검증용 로더는 섞지 않고(`shuffle=False`) 자투리도 버리지 않습니다.

**2. 사전 학습된 모델 불러오기**
```python
CHOOSE_MODEL = "gpt2-medium (355M)"
BASE_CONFIG.update(model_configs[CHOOSE_MODEL])
model_name = "gpt2-medium-355M.pth"
model = load_gpt2_model(model_name, BASE_CONFIG)
model.eval()
model.to(device)
```
이번에는 **GPT-2 Medium**(약 3억 5천만 개 가중치, 24층, 16헤드, 벡터 길이 1024)을 씁니다. 지시를 이해하고 문장으로 답하는 것은 스팸 분류보다 어려운 일이라 더 큰 모델이 필요합니다. Chapter 6 과 달리 **출력 헤드를 바꾸지 않고, 아무것도 동결하지 않습니다.**

**3. 미세 조정**
```python
with torch.no_grad():
    train_loss = calc_loss_loader(train_loader, model, device, num_batches=5)
    val_loss = calc_loss_loader(val_loader, model, device, num_batches=5)
```
학습 **전** 손실을 먼저 재 둡니다. 나중에 얼마나 좋아졌는지 비교하기 위한 기준점입니다.

```python
optimizer = torch.optim.AdamW(model.parameters(), lr=0.00005, weight_decay=0.1)
num_epochs = 2
train_losses, val_losses, tokens_seen = train_model_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=5, eval_iter=5,
    start_context=format_input(val_data[0]), tokenizer=tokenizer
)
```
- **Chapter 5 의 학습 루프(`train_model_simple`)를 그대로** 사용합니다. 데이터의 생김새만 바뀌었을 뿐 "다음 토큰 맞히기"라는 학습 방식은 같기 때문입니다.
- 학습률은 작게(0.00005), 에포크는 2번만. 이미 잘 학습된 모델이라 조금만 고치면 됩니다.
- `start_context=format_input(val_data[0])` : 에포크가 끝날 때마다 검증 데이터 첫 항목의 프롬프트로 답변을 생성해 보여 줍니다.

**4. 테스트 데이터에 답변 생성하기**
```python
for i, entry in tqdm(enumerate(test_data), total=len(test_data)):
    input_text = format_input(entry)
    token_ids = generate(
        model=model,
        idx=text_to_token_ids(input_text, tokenizer).to(device),
        max_new_tokens=256,
        context_size=BASE_CONFIG["context_length"],
        eos_id=50256
    )
    generated_text = token_ids_to_text(token_ids, tokenizer)
    response_text = generated_text[len(input_text):].replace("### Response:", "").strip()
    test_data[i]["model_response"] = response_text
```

| 코드 | 설명 |
|---|---|
| `tqdm(..., total=...)` | 반복문을 감싸면 진행률 막대가 표시됨. `total` 은 전체 개수 |
| `enumerate(test_data)` | (번호, 항목)을 함께 꺼냄 |
| `format_input(entry)` | 학습 때와 **같은 양식**의 프롬프트. 단, 답변은 붙이지 않음 → 모델이 이어서 쓰게 함 |
| `eos_id=50256` | "글 끝" 토큰이 나오면 **생성을 멈춤**. `custom_collate_fn` 에서 첫 50256 을 정답으로 남겨 둔 덕분에 모델이 답을 마치면 이 토큰을 내놓습니다 |
| `max_new_tokens=256` | 끝 토큰이 안 나와도 최대 256개까지만 |
| `generated_text[len(input_text):]` | 생성 결과는 "프롬프트 + 답변" 전체이므로, **프롬프트 길이만큼 앞을 잘라** 답변만 남김 |
| `.replace("### Response:", "")` | 표지판 글자 제거 |
| `.strip()` | 앞뒤 공백·줄바꿈 제거 |
| `test_data[i]["model_response"] = ...` | 각 항목 딕셔너리에 **새 키**를 추가해 모델의 답을 저장 |

**5. 저장**
```python
with open(test_data_path, "w") as file:
    json.dump(test_data, file, indent=4)
```
모델의 답이 추가된 테스트 데이터를 JSON 파일로 저장합니다. `indent=4` 는 사람이 읽기 좋게 4칸 들여쓰기. 이 파일에서 `output`(모범 답)과 `model_response`(모델의 답)를 나란히 비교해 볼 수 있습니다.

```python
file_name = f"outputs/{re.sub(r'[ ()]', '', CHOOSE_MODEL) }-sft-standalone.pth"
torch.save(model.state_dict(), file_name)
```
- `re.sub(패턴, 바꿀 글자, 원본)` : 패턴에 맞는 글자를 찾아 바꿉니다. `r'[ ()]'` 는 "공백, `(`, `)` 중 아무거나"라는 뜻이고 `''`(빈 문자열)로 바꾸므로 **지웁니다.**
  `"gpt2-medium (355M)"` → `"gpt2-medium355M"` → 파일 이름 `gpt2-medium355M-sft-standalone.pth`
- 파일 이름에 공백이나 괄호가 있으면 다루기 불편해서 제거하는 것입니다.
- `torch.save(model.state_dict(), ...)` : 미세 조정된 가중치를 저장합니다.

(주석 중 "JSON 파일로 저S장" 은 "저장"의 오타입니다.)

In [1]:
# 1
# Copyright (c) Sebastian Raschka under Apache License 2.0 (see LICENSE.txt).
# Source for "Build a Large Language Model From Scratch"
#   - https://www.manning.com/books/build-a-large-language-model-from-scratch
# Code: https://github.com/rasbt/LLMs-from-scratch
#
# 7장의 코드를 기반으로 한 최소한의 Instruction Finetuning 예제 파일

from functools import partial
from importlib.metadata import version
import json
import os
import re
import time

import matplotlib.pyplot as plt
import requests
import tiktoken
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

# 로컬 파일(이전 챕터에서 작성된 유틸리티 함수들)에서 필요한 모듈 임포트
# GPTModel 구조, 생성 함수, 학습 루프 함수 등을 가져옵니다.
from previous_chapters import (
    calc_loss_loader,
    generate,
    GPTModel,
    text_to_token_ids,
    train_model_simple,
    token_ids_to_text,
    load_gpt2_model
)

class InstructionDataset(Dataset):
    """
    지시사항(Instruction) 데이터셋을 처리하는 PyTorch Dataset 클래스.
    입력 데이터(지시+입력)와 정답 데이터(응답)를 하나의 텍스트로 합쳐서 토큰화합니다.
    """
    def __init__(self, data, tokenizer):
        self.data = data

        # 텍스트 데이터 미리 토큰화 (Pre-tokenize)
        self.encoded_texts = []
        for entry in data:
            # 포맷팅 함수를 이용해 "지시사항 + 입력" 텍스트 생성
            instruction_plus_input = format_input(entry)
            # 정답(Response) 텍스트 생성
            response_text = f"\n\n### Response:\n{entry['output']}"
            
            # 모델은 이 전체 텍스트(질문+답변)를 보고 다음 토큰을 예측하도록 학습됨
            full_text = instruction_plus_input + response_text
            self.encoded_texts.append(
                tokenizer.encode(full_text)
            )

    def __getitem__(self, index):
        return self.encoded_texts[index]

    def __len__(self):
        return len(self.data)


def custom_collate_fn(
    batch,
    pad_token_id=50256,
    ignore_index=-100,
    allowed_max_length=None,
    device="cpu"
):
    """
    데이터 로더에서 배치를 만들 때 사용하는 커스텀 함수.
    가변 길이의 시퀀스를 배치의 최대 길이에 맞춰 패딩(padding)하고, 
    정답(target) 데이터에서 패딩 부분은 손실(loss) 계산에서 제외하도록 처리합니다.
    """
    # 배치 내에서 가장 긴 시퀀스 길이 계산 (패딩을 위해 +1 여유)
    batch_max_length = max(len(item)+1 for item in batch)

    # 입력(inputs)과 정답(targets) 리스트 준비
    inputs_lst, targets_lst = [], []

    for item in batch:
        new_item = item.copy()
        # 문장 끝 토큰 <|endoftext|> 추가
        # TODO: 시퀀스 끝에 추가할 토큰 값을 채우세요.
        # 힌트: 패딩 전에 구분 토큰이 필요한 경우 pad_token_id를 추가합니다.
        # new_item += [????]
        new_item += [pad_token_id]
        
        # 가장 긴 길이에 맞춰 패딩 추가
        # TODO: 패딩 토큰 값을 채우세요.
        # 힌트: 모든 샘플 길이를 맞추기 위해 pad_token_id로 채웁니다.
        # padded = new_item + [????] * (batch_max_length - len(new_item))
        padded = new_item + [pad_token_id] * (batch_max_length - len(new_item))
        
        # TODO: 입력과 타깃 슬라이싱 인덱스를 채우세요.
        # 힌트: 다음 토큰 예측에서는 inputs=padded[:-1], targets=padded[1:] 형태를 만듭니다.
        # 입력은 마지막 토큰 제외 (0 ~ n-1)
        # inputs = torch.tensor(padded[:????]) 
        inputs = torch.tensor(padded[:-1]) 
        # 정답은 첫 번째 토큰 제외 (1 ~ n) -> 다음 토큰 예측 문제이므로
        # targets = torch.tensor(padded[????:]) 
        targets = torch.tensor(padded[1:]) 

        # 중요: 패딩 부분 마스킹 처리
        # 타겟에서 패딩 토큰인 부분은 ignore_index(-100)로 바꿔서 CrossEntropyLoss 계산 시 무시되게 함
        mask = targets == pad_token_id
        indices = torch.nonzero(mask).squeeze()
        
        # 패딩이 시작되는 지점 이후의 모든 타겟 값을 -100으로 설정
        # region [패딩 마스킹 처리]
        # TODO: 패딩 무시 시작 인덱스를 채우세요.
        # 힌트: 첫 번째 패딩은 유지하고 그 뒤의 패딩부터 ignore_index로 설정합니다.
        if indices.numel() > 1:
            # targets[indices[????:]] = ignore_index
            targets[indices[1:]] = ignore_index
        # endregion
        
        # 선택적으로 최대 시퀀스 길이 제한 (메모리 관리 등 목적)
        if allowed_max_length is not None:
            inputs = inputs[:allowed_max_length]
            targets = targets[:allowed_max_length]

        inputs_lst.append(inputs)
        targets_lst.append(targets)

    # 텐서로 변환 및 디바이스(GPU/CPU)로 이동
    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)

    return inputs_tensor, targets_tensor


def download_and_load_file(file_path, url):
    """
    데이터 파일이 없으면 다운로드하고, JSON 형태로 로드하는 유틸리티 함수
    """
    if not os.path.exists(file_path):
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        text_data = response.text
        with open(file_path, "w", encoding="utf-8") as file:
            file.write(text_data)

    with open(file_path, "r", encoding="utf-8") as file:
        data = json.load(file)

    return data


def format_input(entry):
    """
    Alpaca 스타일의 프롬프트 템플릿을 적용하는 함수.
    모델에게 역할을 부여하고 입력 형식을 통일합니다.
    """
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    # 추가적인 입력 정보(Context 등)가 있는 경우 추가
    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""

    return instruction_text + input_text


def plot_losses(epochs_seen, tokens_seen, train_losses, val_losses):
    """
    학습 진행 상황(Loss)을 그래프로 시각화하여 PDF로 저장하는 함수
    """
    fig, ax1 = plt.subplots(figsize=(12, 6))

    # 에폭(Epoch) 기준 Loss 그래프
    ax1.plot(epochs_seen, train_losses, label="Training loss")
    ax1.plot(epochs_seen, val_losses, linestyle="-.", label="Validation loss")
    ax1.set_xlabel("Epochs")
    ax1.set_ylabel("Loss")
    ax1.legend(loc="upper right")

    # 처리한 토큰 수(Tokens seen) 기준 보조 x축 생성
    ax2 = ax1.twiny() 
    ax2.plot(tokens_seen, train_losses, alpha=0) 
    ax2.set_xlabel("Tokens seen")

    fig.tight_layout()
    plot_name = "outputs/loss-plot-standalone.pdf"
    print(f"Plot saved as {plot_name}")
    plt.savefig(plot_name)
    # plt.show()


def main(test_mode=False):
    #######################################
    # 패키지 버전 출력 (디버깅용)
    #######################################
    print()
    pkgs = [
        "matplotlib",  # 시각화
        "tiktoken",    # 토크나이저
        "torch",       # 딥러닝 프레임워크
        "tqdm",        # 진행률 표시 바
    ]
    for p in pkgs:
        print(f"{p} version: {version(p)}")
    print(50*"-")

    #######################################
    # 1. 데이터셋 다운로드 및 준비
    #######################################
    file_path = "datas/instruction-data.json"
    url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch07/01_main-chapter-code/instruction-data.json"
    data = download_and_load_file(file_path, url)

    # 데이터 분할 (Train 85% / Test 10% / Valid 5%)
    train_portion = int(len(data) * 0.85) 
    test_portion = int(len(data) * 0.1)   

    train_data = data[:train_portion]
    test_data = data[train_portion:train_portion + test_portion]
    val_data = data[train_portion + test_portion:]

    # 테스트 모드일 경우 아주 적은 데이터만 사용 (빠른 실행 확인용)
    if test_mode:
        train_data = train_data[:10]
        val_data = val_data[:10]
        test_data = test_data[:10]

    print("Training set length:", len(train_data))
    print("Validation set length:", len(val_data))
    print("Test set length:", len(test_data))
    print(50*"-")

    # 토크나이저 및 디바이스 설정
    tokenizer = tiktoken.get_encoding("gpt2")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device:", device)
    print(50*"-")

    # 커스텀 collate 함수 설정 (partial을 사용해 고정 인자 전달)
    customized_collate_fn = partial(custom_collate_fn, device=device, allowed_max_length=1024)

    num_workers = 0
    batch_size = 8
    torch.manual_seed(123)

    # 학습용 데이터 로더
    train_dataset = InstructionDataset(train_data, tokenizer)
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        collate_fn=customized_collate_fn, # 커스텀 배치 처리 함수 사용
        shuffle=True,
        drop_last=True,
        num_workers=num_workers
    )

    # 검증용 데이터 로더
    val_dataset = InstructionDataset(val_data, tokenizer)
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        collate_fn=customized_collate_fn,
        shuffle=False,
        drop_last=False,
        num_workers=num_workers
    )

    #######################################
    # 2. 사전 학습된 모델 로드 (Pre-trained Model)
    #######################################

    if test_mode:
        # 테스트 모드용 초소형 더미 모델
        BASE_CONFIG = {
            "vocab_size": 50257,
            "context_length": 120,
            "drop_rate": 0.0,
            "qkv_bias": False,
            "emb_dim": 12,
            "n_layers": 1,
            "n_heads": 2
        }
        model = GPTModel(BASE_CONFIG)
        model.eval()
        device = "cpu"
        CHOOSE_MODEL = "Small test model"

    else:
        # 실제 학습용 설정: GPT-2 Medium (355M)
        BASE_CONFIG = {
            "vocab_size": 50257,     
            "context_length": 1024,  
            "drop_rate": 0.0,        
            "qkv_bias": True         
        }

        model_configs = {
            "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
            "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
            "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
            "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
        }

        CHOOSE_MODEL = "gpt2-medium (355M)"
        BASE_CONFIG.update(model_configs[CHOOSE_MODEL])

        # 로컬에 저장된 모델 가중치 파일 로드
        model_name = "gpt2-medium-355M.pth"
        model = load_gpt2_model(model_name, BASE_CONFIG)

        model.eval()
        model.to(device)

    print("Loaded model:", CHOOSE_MODEL)
    print(50*"-")

    #######################################
    # 3. 모델 미세 조정 (Finetuning)
    #######################################
    print("Initial losses")
    # 학습 전 초기 손실값 확인
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=5)
        val_loss = calc_loss_loader(val_loader, model, device, num_batches=5)

    print("   Training loss:", train_loss)
    print("   Validation loss:", val_loss)

    start_time = time.time()
    # 옵티마이저 설정 (AdamW)
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.00005, weight_decay=0.1)

    num_epochs = 2

    torch.manual_seed(123)
    
    # 실제 학습 수행 (previous_chapters.py에 정의된 함수 사용)
    train_losses, val_losses, tokens_seen = train_model_simple(
        model, train_loader, val_loader, optimizer, device,
        num_epochs=num_epochs, eval_freq=5, eval_iter=5,
        start_context=format_input(val_data[0]), tokenizer=tokenizer
    )

    end_time = time.time()
    execution_time_minutes = (end_time - start_time) / 60
    print(f"Training completed in {execution_time_minutes:.2f} minutes.")

    # 학습 결과 시각화
    epochs_tensor = torch.linspace(0, num_epochs, len(train_losses))
    plot_losses(epochs_tensor, tokens_seen, train_losses, val_losses)
    print(50*"-")

    #######################################
    # 4. 결과 저장 및 테스트
    #######################################
    print("Generating responses")
    # 테스트 데이터셋에 대해 모델 응답 생성
    for i, entry in tqdm(enumerate(test_data), total=len(test_data)):

        input_text = format_input(entry)

        # 모델 생성 (Inference)
        token_ids = generate(
            model=model,
            idx=text_to_token_ids(input_text, tokenizer).to(device),
            max_new_tokens=256,
            context_size=BASE_CONFIG["context_length"],
            eos_id=50256
        )
        generated_text = token_ids_to_text(token_ids, tokenizer)
        
        # 프롬프트 부분을 제거하고 응답 부분만 추출
        response_text = generated_text[len(input_text):].replace("### Response:", "").strip()

        test_data[i]["model_response"] = response_text

    # 생성된 응답을 포함하여 JSON 파일로 저S장
    test_data_path = "outputs/instruction-data-with-response-standalone.json"
    with open(test_data_path, "w") as file:
        json.dump(test_data, file, indent=4) 
    print(f"Responses saved as {test_data_path}")

    # 미세 조정된 모델 가중치 저장
    file_name = f"outputs/{re.sub(r'[ ()]', '', CHOOSE_MODEL) }-sft-standalone.pth"
    torch.save(model.state_dict(), file_name)
    print(f"Model saved as {file_name}")



---
## 셀 2 — 실행

### 🎯 이 셀이 하는 일
```python
main(test_mode=False)
```
셀 1에서 정의한 `main` 을 **실제 모드**로 실행합니다.

- `test_mode=False` : 전체 데이터(935개)와 GPT-2 Medium 으로 진짜 학습을 합니다. GPU 에서도 몇 분이 걸립니다.
- `test_mode=True` 로 바꾸면 : 데이터 10개 + 아주 작은 가짜 모델로 **코드에 오류가 없는지만** 금방 확인합니다(결과 글은 엉터리입니다). 코드를 수정한 뒤 처음 돌려 볼 때 유용합니다.

> 실행 전에 맨 위 안내의 **폴더 이름(`datas/`, `outputs/`)** 을 꼭 확인하세요.

### 실행 결과 읽기
저장되어 있는 출력은 학습 초반까지만 남아 있습니다.
```
Training set length: 935
Validation set length: 55
Test set length: 110
Device: cuda
Loaded model: gpt2-medium (355M)
Initial losses
   Training loss: 3.8390541553497313
   Validation loss: 3.761933517456055
Ep 1 (Step 000000): Train loss 2.637, Val loss 2.626
Ep 1 (Step 000005): Train loss 1.174, Val loss 1.102
```
- 데이터가 935 / 55 / 110 개로 나뉘었습니다(85% / 5% / 10%).
- **학습 전 손실이 약 3.8** 입니다. Chapter 5 에서 무작위 모델의 시작 손실이 약 10 이었던 것과 비교해 보세요. 이미 영어를 아는 모델이라 출발점부터 훨씬 낮습니다.
- 단 **6스텝**(배치 6개) 만에 손실이 1.1 까지 떨어집니다. 모델이 "새로운 지식"을 배우는 게 아니라 **"이 양식에는 이렇게 답한다"는 형식**만 익히면 되기 때문에 매우 빨리 적응합니다.
- 끝까지 실행하면 5스텝마다 손실이 출력되고, 에포크가 끝날 때마다 샘플 답변이 한 번씩 출력된 뒤, 테스트 110개에 대한 답변 생성 진행률 막대와 저장 메시지가 나옵니다.

(이 셀 아래의 빈 코드 셀은 원본 노트북에 있던 것으로, 내용이 없습니다.)

In [ ]:
# 2
main(test_mode=False)



matplotlib version: 3.10.8
tiktoken version: 0.12.0
torch version: 2.11.0+cu128
tqdm version: 4.67.3
--------------------------------------------------
Training set length: 935
Validation set length: 55
Test set length: 110
--------------------------------------------------
Device: cuda
--------------------------------------------------
Loaded model: gpt2-medium (355M)
--------------------------------------------------
Initial losses
   Training loss: 3.8390541553497313
   Validation loss: 3.761933517456055
Ep 1 (Step 000000): Train loss 2.637, Val loss 2.626
Ep 1 (Step 000005): Train loss 1.174, Val loss 1.102


---
## 📌 Chapter 7 정리

```
 {instruction, input, output}
        |  format_input                       fixed prompt template
        v
 "### Instruction: ... ### Input: ... ### Response: ..."
        |  tokenize                            (InstructionDataset)
        v
 custom_collate_fn :  add EOS -> pad -> shift by 1 -> extra pads = -100
        |
        v
 train_model_simple (same loop as Chapter 5)
        |
        v
 generate(..., eos_id=50256)  ->  cut off the prompt  ->  model_response
```

**꼭 기억할 4가지**
1. 지시 미세 조정 = **"프롬프트 + 모범 답변"을 한 글로 이어 붙여 다음 토큰 맞히기**로 학습. 학습 루프는 Chapter 5 와 같다.
2. **프롬프트 양식**은 학습 때와 사용할 때 똑같아야 한다.
3. 패딩 자리의 정답은 **−100** 으로 바꿔 채점에서 뺀다. 단 **첫 번째 끝 토큰은 남겨** "멈추는 법"을 배우게 한다.
4. 생성할 때 `eos_id` 를 주면 답이 끝나는 곳에서 멈춘다.

## 🎓 Chapter 2 ~ 7 전체 흐름

| 챕터 | 한 일 |
|---|---|
| 2 | 글 → 토큰 → (입력, 정답) 배치 → 임베딩 |
| 3 | 어텐션 : 토큰끼리 정보를 주고받는 장치 |
| 4 | GPT 모델 조립 + 글 생성 |
| 5 | 사전 학습 : 다음 토큰 맞히기로 가중치 학습 |
| 6 | 미세 조정 ① : 분류기로 고쳐 쓰기 (+ LoRA) |
| 7 | 미세 조정 ② : 지시에 답하는 모델로 만들기 |